### PINN Mesh approach

This is a discarded approach due to the bad sampling both of the internal and boundary points. The problem lies in the mesh generation which puts to few points on the boundary while having too many in the domain; this, combained with the limited RAM, makes the training painfully slow as well as difficult to refine. However, for archival purposes, the code hasn't been deleted though it can be safely ignored.

In [ ]:
# Geometry definition
aMax  = 1e-3 # Maximum FE area
L     = 1   # Squre domain length
order = 2   # Order of FE

domain = {'SquareEdge': L, \
          'VerticesBoundaryCondition': [1,1,1,1], \
          'EdgesBoundaryCondition': [1,1,1,1], \
          'DiscretizationType': 1, \
          'MeshCellsMaximumArea': aMax}
[meshInfo, mesh] = gedim.CreateDomainSquare(domain,lib)

discreteSpace = {'Order': order, \
                 'Type': 1, \
                 'BoundaryConditionsType': [1,2]}
[problemData, dofs, strongs] = gedim.Discretize(discreteSpace,lib)


# Mesh boundary points
xBC = torch.from_numpy(strongs[0,:].reshape(-1,1)).float().to(device)
yBC = torch.from_numpy(strongs[1,:].reshape(-1,1)).float().to(device)

uBCH  = torch.full((strongs.shape[1],1),0,dtype=torch.float,device=device) # Homogenous BC

# Mesh internal points
xIP = torch.from_numpy(dofs[0,:].reshape(-1,1)).float().to(device)
xIP.requires_grad = True
yIP = torch.from_numpy(dofs[1,:].reshape(-1,1)).float().to(device)
yIP.requires_grad = True

zeros = torch.full((dofs.shape[1],1),0,dtype=torch.float,device=device)

In [ ]:
seed = 0 # Set a seed for random number generation for reproducibility sake
torch.manual_seed(seed) # Without it, it'd be random every time the code is run
net  = PINN().to(device) # Model

MSELoss   = torch.nn.MSELoss() # Mean squared error loss function
optimiser = torch.optim.Adam(net.parameters())
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser,factor=0.75,patience=5000)

In [ ]:
# Training/Fitting phase
l = 1 # Lambda to balance the loss functions

tol   = 1e-7
nEMax = int(1e5) # Number of maximum epochs
e     = 0
loss  = torch.tensor(1)

loss100   = np.zeros((100,1))
mseBC100  = np.zeros((100,1))
mseRes100 = np.zeros((100,1))

mseResRif = 0

while aMax >= tol and e < nEMax and loss.item() > tol:
    optimiser.zero_grad() # Set the gradients to zero

    # Random boundary parameters
    [m0,m1] = RandomParam(strongs.shape[1],0.1,1)

    # BC Loss
    uBCP = net(xBC,yBC,m0,m1)
    mseBC = MSELoss(uBCP,uBCH)


    # Random internal parameters
    [m0,m1] = RandomParam(dofs.shape[1],0.1,1)

    # PDE Loss
    resIP = ResPDE(xIP,yIP,m0,m1,net) # Internal point residual
    mseRes = MSELoss(resIP,zeros)


    epsilon = 1e-8 # Small number to prevent division by zero
    r = mseBC.item()/(mseRes.item() + epsilon)
    l = 1/(1 + r)
    loss = (1-l)*mseBC + l*mseRes

    loss.backward()  # Backpropagation
    optimiser.step() # Optimiser update
    scheduler.step(loss.item()) # Scheduler check

    loss100[e % 100]   = loss.item()
    mseRes100[e % 100] = mseRes.item()
    mseBC100[e % 100]  = mseBC.item()

    if e % 100 == 0:

        meanL   = sum(loss100)[0]/100
        meanBC  = sum(mseBC100)[0]/100
        meanRes = sum(mseRes100)[0]/100

        with torch.autograd.no_grad():
            # sys.stdout.flush()
            # print(f"\r{e} Loss: {loss.item():.16f}",end="")
            print(e,
                "\tmeanL:",   '{:.4e}'.format(meanL), \
                "\tmeanBC:" , '{:.6e}'.format(meanBC), \
                "\tmeanRes:", '{:.6e}'.format(meanRes), \
                #   "\tmseDiff:",'{:.2e}'.format(mseRelDiff), \
                #   "\tdiff:",   '{:.6e}'.format(diff), \
                "\tlr:",     '{:.0e}'.format(optimiser.param_groups[0]['lr']), \
                "\tλ:",      '{:.4f}'.format(l))
            
        if meanL <= 1e-4:
            aMax /= 2
            domain['MeshCellsMaximumArea'] = aMax
            
            [meshInfo, mesh] = gedim.CreateDomainSquare(domain,lib)
            [problemData, dofs, strongs] = gedim.Discretize(discreteSpace,lib)

            # Mesh boundary points
            xBC = torch.from_numpy(strongs[0,:].reshape(-1,1)).float().to(device)
            yBC = torch.from_numpy(strongs[1,:].reshape(-1,1)).float().to(device)

            uBCH  = torch.full((strongs.shape[1],1),0,dtype=torch.float,device=device) # Homogenous BC

            # Mesh internal points
            xIP = torch.from_numpy(dofs[0,:].reshape(-1,1)).float().to(device)
            xIP.requires_grad = True
            yIP = torch.from_numpy(dofs[1,:].reshape(-1,1)).float().to(device)
            yIP.requires_grad = True

            zeros = torch.full((dofs.shape[1],1),0,dtype=torch.float,device=device)

            optimiser.param_groups[0]['lr'] = 1e-3

    e += 1

### PINN plot

The following code describe a way to plot the PINN solution using the method »np.meshgrid(x,y)»; it's very involved and long, especially compared to just using the GeDiM mesh after converting the PINN solution in an array with the function «buildSolPINN».

In [ ]:
x = np.arange(0,1,0.02)
y = np.arange(0,1,0.02)
[msx, msy] = np.meshgrid(x,y)

# Reshape the previous matricies, «msx» and «msy», in two column
# vector; -1 is an arbitrary [dimensionally coherent] number
x = np.ravel(msx).reshape(-1,1)
y = np.ravel(msy).reshape(-1,1)

# x = Variable(torch.from_numpy(x).float(),requires_grad=True)
# y = Variable(torch.from_numpy(y).float(),requires_grad=True)

x = torch.from_numpy(x).float().to(device)
y = torch.from_numpy(y).float().to(device)

x.requires_grad = True
y.requires_grad = True

m0 = torch.full((2500,1),m0,dtype=torch.float,device=device)
m1 = torch.full((2500,1),m1,dtype=torch.float,device=device)

net.eval()
with torch.no_grad():
    u = net(x,y,m0,m1)
    u = u.data.cpu().numpy()

msu  = u.reshape(msx.shape)
surf = ax1.plot_surface(msx,msy,msu,cmap=cm.coolwarm,\
                       linewidth=0,antialiased=False)
fig.colorbar(surf,shrink=0.3,aspect=10,pad=0.1)

ax1.set_title("PINN Solution")